# Análisis más detallado de un VAE
Vamos a usar el vae visto en `0.5_VAEs&GANs` para profundizar en algunas cuestiones referidas a la función de costo y a la forma de las distribuciones. 

### Seteos iniciales

In [67]:
import os
import sys
sys.path.append("..")
from dataclasses import dataclass, field
#from pathlib import Path
from typing import Literal
import einops
import torch as t
import torchinfo
from datasets import load_dataset
from einops.layers.torch import Rearrange
from jaxtyping import Float
from torch import Tensor, nn
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms
from tqdm import tqdm
from plotly_utils import imshow
import utils
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Módulos de la parte de CNNs
from part2_cnns.solutions import BatchNorm2d, Conv2d, Linear, ReLU, Sequential

device = t.device(
    "cuda:1"
    if t.cuda.device_count() > 1
    else ("cuda:0" if t.cuda.is_available() else "cpu")
)
print(f"Usando dispositivo: {device}")

Usando dispositivo: cuda:0


### Funciones auxiliares

In [ ]:
def get_dataset(train: bool = True) -> Dataset:
    img_size = 28
    transform = transforms.Compose(
        [
            transforms.Resize(img_size),
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,)),
        ]
    )
    trainset = datasets.MNIST(
        root="./data",
        transform=transform,
        download=True,
        train=train,
    )
    return trainset

def display_data(x: Tensor, nrows: int, title: str) -> None:
    """Displays a batch of data, using plotly."""
    ncols = x.shape[0] // nrows
    # Reshape into the right shape for plotting (make it 2D if image is monochrome)
    y = einops.rearrange(x, "(b1 b2) c h w -> (b1 h) (b2 w) c", b1=nrows).squeeze()
    # Normalize in the 0-1 range, then map to integer type
    y = (y - y.min()) / (y.max() - y.min())
    y = (y * 255).to(dtype=t.uint8)
    # Display data
    imshow(
        y,
        binary_string=(y.ndim == 2),
        height=50 * (nrows + 4),
        width=50 * (ncols + 5),
        title=f"{title}<br>single input shape = {x[0].shape}",
    )

def create_grid_of_latents(
    model: nn.Module,
    interpolation_range: tuple[float, float] = (-1, 1),
    n_points: int = 11,
    dims: tuple[int, int] = (0, 1),
    ) -> Float[Tensor, "rows_x_cols latent_dims"]:
    """Create a tensor of zeros which varies along the 2 specified dimensions of the latent space."""
    grid_latent = t.zeros(n_points, n_points, model.latent_dim_size, device=device)
    x = t.linspace(*interpolation_range, n_points)
    grid_latent[..., dims[0]] = x.unsqueeze(-1)  # rows vary over dim=0
    grid_latent[..., dims[1]] = x  # cols vary over dim=1
    return grid_latent.flatten(0, 1)  # flatten over (rows, cols) into a single batch dimension

### Visualización de los datos a utilizar

In [ ]:
# Ejemplo de un batch de imágenes
trainset = get_dataset()
x = next(iter(DataLoader(trainset, batch_size=25)))[0]
display_data(x, nrows=5, title="MNIST data")

# Ejemplo de 10 digitos de test,
# Creamos un tensor de 10 imágenes con los digitos 0 a 9 llamado HOlDOUT_DATA [10,1,H,W] o sea b=10
testset = get_dataset(train=False)
HOLDOUT_DATA = dict()
for data, target in DataLoader(testset, batch_size=1):
    if target.item() not in HOLDOUT_DATA:
        HOLDOUT_DATA[target.item()] = data.squeeze()
        if len(HOLDOUT_DATA) == 10:
            break
HOLDOUT_DATA = t.stack([HOLDOUT_DATA[i] for i in range(10)]).to(dtype=t.float, device=device).unsqueeze(1)
display_data(HOLDOUT_DATA, nrows=1, title="MNIST holdout data")

### Definición del modelo VAE

In [29]:
class VAE(nn.Module):
    encoder: nn.Module
    decoder: nn.Module

    def __init__(self, latent_dim_size: int, hidden_dim_size: int):
        super().__init__()
        self.latent_dim_size = latent_dim_size
        self.hidden_dim_size = hidden_dim_size
        self.encoder = Sequential(
            Conv2d(in_channels=1,out_channels= 16, kernel_size=4, stride=2, padding=1),
            ReLU(),
            Conv2d(in_channels=16,out_channels= 32, kernel_size=4, stride=2, padding=1),
            ReLU(),
            Rearrange('b c h w -> b (c h w)'),
            Linear(32*7*7,self.hidden_dim_size),
            ReLU(),       
            Linear(hidden_dim_size,latent_dim_size * 2),
            Rearrange('b (n lat_dim) -> n b lat_dim',n=2)
        )
        self.decoder = self.decoder = nn.Sequential(
            Linear(latent_dim_size, hidden_dim_size),
            ReLU(),
            Linear(hidden_dim_size, 7 * 7 * 32),
            ReLU(),
            Rearrange("b (c h w) -> b c h w", c=32, h=7, w=7),
            nn.ConvTranspose2d(32, 16, 4, stride=2, padding=1, bias=False),
            ReLU(),
            nn.ConvTranspose2d(16, 1, 4, stride=2, padding=1, bias=False),
        )

    def sample_latent_vector(
        self, x: Float[Tensor, "batch 1 height width"]
    ) -> tuple[
        Float[Tensor, "batch latent"],
        Float[Tensor, "batch latent"],
        Float[Tensor, "batch latent"],
    ]:
        """
        Passes `x` through the encoder, returns tuple of (sampled latent vector, mean, log std dev).
        This function can be used in `forward`, but also used on its own to generate samples for
        evaluation.
        """
        mu, logsigma = self.encoder(x)
        sigma = t.exp(logsigma) 
        z = mu + sigma * t.randn_like(sigma)
        return z, mu, logsigma
        

    def forward(
        self, x: Float[Tensor, "batch 1 height width"]
    ) -> tuple[
        Float[Tensor, "batch 1 height width"],
        Float[Tensor, "batch latent"],
        Float[Tensor, "batch latent"],
    ]:
        """
        Passes `x` through the encoder and decoder. Returns the reconstructed input, as well as mu
        and logsigma.
        """

        z, mu, logsigma = self.sample_latent_vector(x)
        x_prima = self.decoder(z)
        return x_prima, mu, logsigma


### Entrenamiento del modelo VAE

In [ ]:
@dataclass
class VAEArgs:
    # architecture
    latent_dim_size: int = 5
    hidden_dim_size: int = 128
    beta_kl: float = 0.1

    # data / training
    batch_size: int = 512
    epochs: int = 10
    lr: float = 1e-3
    betas: tuple[float, float] = (0.5, 0.999)


class VAETrainer:
    def __init__(self, args: VAEArgs):
        self.args = args
        self.trainset = get_dataset()
        self.trainloader = DataLoader(self.trainset, batch_size=args.batch_size, shuffle=True, num_workers=8)
        self.model = VAE(
            latent_dim_size=args.latent_dim_size,
            hidden_dim_size=args.hidden_dim_size,
        ).to(device)
        self.optimizer = t.optim.Adam(self.model.parameters(), lr=args.lr, betas=args.betas)

    def training_step(
        self, img: Float[Tensor, "batch 1 height width"]
        ) -> Float[Tensor, ""]:
        """
        Performs a training step on the batch of images in `img`. Returns the loss. Logs to wandb
        if enabled.
        """
        # Get the different loss components, as well as the total loss
        img = img.to(device)
        img_reconstructed, mu, logsigma = self.model(img)
        reconstruction_loss = nn.MSELoss()(img, img_reconstructed)
        kl_div_loss = (0.5 * (mu**2 + t.exp(2 * logsigma) - 1) - logsigma).mean() * self.args.beta_kl
        loss = reconstruction_loss + kl_div_loss

        # Backprop on the loss, and step with optimizers
        loss.backward()
        self.optimizer.step()
        self.optimizer.zero_grad()

        return loss

    @t.inference_mode() # esto le dice a log_samples que ponga todo en un with que no toque los gradientes
    def log_samples(self) -> None:
        """
        Evaluates model on holdout data, either logging to wandb or displaying output inline.
        """
        assert self.step > 0, "First call should come after a training step. Remember to increment `self.step`."
        output = self.model(HOLDOUT_DATA)[0]
        display_data(t.concat([HOLDOUT_DATA, output]), nrows=2, title="VAE reconstructions")

    def train(self) -> VAE:
        """Performs a full training run."""
        self.step = 0
        for epoch in range(self.args.epochs):
            # Iterate over training data, performing a training step for each batch
            progress_bar = tqdm(self.trainloader, total=int(len(self.trainloader)), ascii=True)
            for img, label in progress_bar:  # remember that label is not used
                img = img.to(device)
                loss = self.training_step(img)
                self.step += 1
                progress_bar.set_description(f"{epoch=:02d}, {loss=:.4f}, batches={self.step:05d}")              
            self.log_samples()

        return self.model
# Entrenamiento
args = VAEArgs(latent_dim_size=5, hidden_dim_size=100)
trainer = VAETrainer(args)
vae = trainer.train() # modelo entrenado listo para usar

### Visualización de la salida ante diferentes vectores latentes
  - Creamos un batch de `(n_points X n_points)` vectores latentes. 
  - Los vectores latentes son distinto de cero solamente en dos dimensiones dadas por la tupla `dims`. 
  - Los valores que adoptan las dimensiones para cada elemento del batch varían en rango de `interpolation_range`
  - eg, para los valores default: el vector latente  de la fila 4 columna 9 (o sea el elemento 36 del batch de 121 elementos) será `v = [0.4,0.6,0,0,0]`. Al aplicar `img = vae.decoder(v)` obtendremos la imagen similar al 9 que se ve en la grilla. 

In [66]:
grid_latent = create_grid_of_latents(vae, interpolation_range=(-1,1), n_points=11, dims=(0,1))
output = vae.decoder(grid_latent)
utils.visualise_output(output, grid_latent, title="VAE latent space visualization")

In [71]:

# Asumo que tenés un modelo VAE entrenado, y un DataLoader/dataset de test con labels
# Ajustá según tus nombres reales de variables
vae.eval()
testset = get_dataset(train=False)
testloader = DataLoader(testset, batch_size=1)
all_mus = []
all_labels = []
with t.inference_mode():
    for img, label in testloader:  # o el nombre que uses para tu dataloader de test
        img = img.to(device)
        z, mu, logvar = encoder(img)  # AJUSTAR según tu implementación
        all_mus.append(mu.cpu())
        all_labels.append(label.cpu())

all_mus = t.cat(all_mus).numpy()       # shape (N, latent_dim_size)
all_labels = t.cat(all_labels).numpy()  # shape (N,)

# Si latent_dim_size == 2, graficamos directo. Si es mayor, reducimos con PCA a 2D
if all_mus.shape[1] > 2:
    all_mus_2d = PCA(n_components=2).fit_transform(all_mus)
else:
    all_mus_2d = all_mus

# Graficar, coloreado por dígito
plt.figure(figsize=(8, 8))
scatter = plt.scatter(all_mus_2d[:, 0], all_mus_2d[:, 1], c=all_labels, cmap="tab10", s=5, alpha=0.6)
plt.colorbar(scatter, label="Dígito")
plt.xlabel("Dimensión latente 1")
plt.ylabel("Dimensión latente 2")
plt.title("Medias μ(x) del encoder, coloreadas por dígito")
plt.show()

# Chequeo numérico: media y std agregados (deberían acercarse a 0 y 1 si el KL hizo su trabajo)
print("Media agregada de μ(x):", all_mus.mean(axis=0))
print("Std agregada de μ(x):", all_mus.std(axis=0))

NameError: name 'encoder' is not defined